## 1. Setup and Configuration

In [ ]:
import sys
from pathlib import Path
import yaml

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.utils.config import load_config
from src.utils.azure_auth import get_ml_client
from src.training.model_loader import (
    download_model_from_foundry,
    register_model_in_workspace,
    verify_model_files,
    download_and_register_model,
    list_available_foundry_models,
)

print("✅ Imports successful")

In [ ]:
# Load configuration
config = load_config()

print(f"Workspace: {config.azure.workspace_name}")
print(f"Resource Group: {config.azure.resource_group}")

In [ ]:
# Load model configuration
model_config_path = project_root / "configs" / "model_config.yaml"
with open(model_config_path) as f:
    model_config = yaml.safe_load(f)

print("\nModel Configuration:")
print(f"  Model: {model_config['model']['name']}")
print(f"  Parameters: {model_config['model']['parameters']}")
print(f"  Context Length: {model_config['model']['context_length']} tokens")
print(f"  Local Path: {model_config['model']['local_path']}")

In [ ]:
# Connect to Azure ML
ml_client = get_ml_client(config.azure)

print(f"✅ Connected to workspace: {ml_client.workspace_name}")

## 2. Browse Available Models

Azure AI Foundry provides several pre-trained models:

In [ ]:
# List common models available in Azure AI Foundry
available_models = list_available_foundry_models(ml_client)

print("Available Models in Azure AI Foundry:\n")
for model in available_models:
    print(f"  - {model}")

## 3. Configure Download Settings

In [ ]:
# Model to download
MODEL_NAME = model_config['model']['name']
LOCAL_PATH = project_root / model_config['model']['local_path']
REGISTERED_NAME = model_config['registration']['name']

print(f"Download Configuration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Local Path: {LOCAL_PATH}")
print(f"  Registered Name: {REGISTERED_NAME}")

# Create output directory
LOCAL_PATH.mkdir(parents=True, exist_ok=True)
print(f"\n✅ Output directory ready: {LOCAL_PATH}")

## 4. Check Disk Space

Phi-4 model requires approximately 30GB of disk space.

In [ ]:
import shutil

# Check available disk space
stats = shutil.disk_usage(LOCAL_PATH.parent)

available_gb = stats.free / (1024**3)
required_gb = 30  # Approximate size for Phi-4

print(f"Disk Space Check:")
print(f"  Available: {available_gb:.2f} GB")
print(f"  Required: {required_gb} GB")

if available_gb < required_gb:
    print(f"\n⚠️  WARNING: Insufficient disk space!")
    print(f"  Free up at least {required_gb - available_gb:.2f} GB before proceeding.")
else:
    print(f"\n✅ Sufficient disk space available")

## 5. Download Model from Azure AI Foundry

⏱️ **This may take 10-30 minutes depending on connection speed.**

**Important Notes:**
- Accept model license terms in Azure AI Foundry portal first
- Ensure you have access to the model in your subscription
- Download progress is logged to console

In [ ]:
import time

print(f"Starting download of '{MODEL_NAME}'...\n")
print("This may take 10-30 minutes. Progress will be logged below.\n")

start_time = time.time()

try:
    model_path = download_model_from_foundry(
        model_name=MODEL_NAME,
        output_dir=LOCAL_PATH,
        ml_client=ml_client,
    )

    elapsed_time = time.time() - start_time

    print(f"\n✅ Download completed successfully!")
    print(f"   Time: {elapsed_time / 60:.2f} minutes")
    print(f"   Location: {model_path}")

except Exception as e:
    print(f"\n❌ Download failed: {e}")
    print(f"\nTroubleshooting:")
    print(f"  1. Accept model license in Azure AI Foundry portal")
    print(f"  2. Verify you have access to the model")
    print(f"  3. Check network connectivity")
    raise

## 6. Verify Model Files

Check that all required files were downloaded correctly.

In [ ]:
# Verify downloaded files
validation = verify_model_files(model_path)

print("Model Validation Results:\n")
print(f"  Valid: {validation['valid']}")
print(f"  Path: {validation['path']}")
print(f"  Total Files: {validation['total_files']}")
print(f"  Total Size: {validation['total_size_mb']:.2f} MB")

print(f"\nFiles Found:")
for file_type, found in validation['files_found'].items():
    status = "✅" if found else "❌"
    print(f"  {status} {file_type}")

if not validation['valid']:
    print(f"\n⚠️  WARNING: Model validation failed!")
    print(f"Some expected files may be missing.")
else:
    print(f"\n✅ Model validation passed!")

## 7. List Downloaded Files

In [ ]:
# Show downloaded files
print(f"Downloaded Files in {model_path}:\n")

for file in sorted(model_path.rglob("*")):
    if file.is_file():
        size_mb = file.stat().st_size / (1024**2)
        rel_path = file.relative_to(model_path)
        print(f"  {rel_path} ({size_mb:.2f} MB)")

## 8. Register Model in Azure ML Workspace

Register the downloaded model so it can be used in training jobs.

In [ ]:
# Register model
registered_model = register_model_in_workspace(
    model_name=REGISTERED_NAME,
    model_path=model_path,
    description=model_config['registration']['description'],
    tags=model_config['registration']['tags'],
    ml_client=ml_client,
)

print(f"\n✅ Model registered successfully!")
print(f"   Name: {registered_model.name}")
print(f"   Version: {registered_model.version}")
print(f"   ID: {registered_model.id}")

## 9. View in Azure ML Studio

In [ ]:
# Generate Azure ML Studio URL
studio_url = (
    f"https://ml.azure.com/model/list"
    f"?wsid=/subscriptions/{config.azure.subscription_id}"
    f"/resourceGroups/{config.azure.resource_group}"
    f"/providers/Microsoft.MachineLearningServices"
    f"/workspaces/{config.azure.workspace_name}"
)

print("View registered models in Azure ML Studio:")
print(studio_url)

## Summary

✅ **Model download and registration complete!**

**What's ready:**
- Phi-4 model downloaded locally
- Model files validated
- Model registered in Azure ML workspace

**Model Details:**
- **Name**: Microsoft Phi-4
- **Parameters**: ~14B
- **Context Length**: 16K tokens
- **License**: Microsoft Research License

## Next Steps

- **Notebook 05**: Fine-tune the model with your training data
- **Notebook 06**: Evaluate fine-tuned model performance

## Alternative: One-Step Download and Register

In [ ]:
# Alternative: Use convenience function for download + register in one step
# Uncomment to use:

# model_path, registered_model = download_and_register_model(
#     model_name=MODEL_NAME,
#     output_dir=LOCAL_PATH,
#     registered_name=REGISTERED_NAME,
#     description=model_config['registration']['description'],
#     tags=model_config['registration']['tags'],
#     ml_client=ml_client,
#     verify=True,
# )
#
# print(f"Model ready: {registered_model.name} v{registered_model.version}")

## Troubleshooting

**License Not Accepted**:
- Go to [Azure AI Foundry](https://ai.azure.com)
- Navigate to Model Catalog → microsoft/phi-4
- Accept license terms

**Access Denied**:
- Verify your Azure subscription has access to AI Foundry
- Check that you have Contributor role on the workspace
- Ensure model is available in your region

**Download Timeout**:
- Increase timeout in model_config.yaml
- Check network connectivity
- Try during off-peak hours

**Insufficient Disk Space**:
- Free up at least 30GB
- Use external storage or cloud storage
- Clean up old model downloads